In [35]:
import json
import re
from datetime import datetime

# -------- Helpers --------

def extract_date(posted_on):
    """
    Extracts date from 'Posted on February 06, 2026 | ...'
    and converts to MM/DD/YYYY
    """
    match = re.search(r"Posted on ([A-Za-z]+ \d{2}, \d{4})", posted_on)
    if not match:
        return None

    date_str = match.group(1)
    return datetime.strptime(date_str, "%B %d, %Y").strftime("%m/%d/%Y")


In [ ]:
# phrases to identify the start of location info in the title.
# here structures location refers to the cases where we can confidently extract a single city name, 
# while unstructured refers to cases where we have multiple locations or ambiguous patterns that 
# prevent us from confidently geocoding to a single city.

TRIGGER_PHRASES = sorted([
    r"Boil Water Advisory Re-issued for",
    r"Boil Water Advisories Rescinded for",
    r"Boil Water Advisory Rescinded for the",
    r"Boil Water Advisory Rescinded for",
    r"Boil Water Advisory Issued for the",
    r"Boil Water Advisory Issued for",
    r"Boil Water Order Rescinded for the",
    r"Boil Water Order Rescinded for",
    r"Boil Water Order Issued for the",
    r"Boil Water Order Issued for",
    r"Boil Water Order for the",
    r"Boil Water Order for",
    r"Boil Water Advisory for the",
    r"Boil Water Advisory for",
], key=len, reverse=True)

# Non-geocodable patterns → these are always unstructured
NON_GEOCODABLE = re.compile(
    r"\b(RWD|PWS|USD|Rural Water|Water District|Co RWD|Subdivision|Unit)\b",
    re.IGNORECASE
)

def extract_city(title):
    if not isinstance(title, str):
        return None, "unmatched"

    city_matches = re.findall(r"City of\s+([^,]+)", title, re.IGNORECASE)
    county_matches = re.findall(r"([A-Za-z\s]+County)", title, re.IGNORECASE)

    # ---- Priority 1: Structured → single "City of XYZ" + single county + no RWD/PWS etc ----
    if len(city_matches) == 1 and len(county_matches) == 1:
        city_name = city_matches[0].strip()
        if not NON_GEOCODABLE.search(city_name):
            return city_name, "structured"

    # ---- Priority 2: Unstructured → trigger phrase pattern ----
    for phrase in TRIGGER_PHRASES:
        pattern = re.compile(re.escape(phrase), re.IGNORECASE)
        match = pattern.search(title)
        if match:
            remainder = title[match.end():].strip()

            # Step 1: Split by comma first
            segments = [s.strip() for s in remainder.split(",")]

            # Step 2: Further split each segment by " and " to catch trailing locations
            expanded_segments = []
            for seg in segments:
                parts = re.split(r"\s+and\s+", seg, flags=re.IGNORECASE)
                expanded_segments.extend([p.strip() for p in parts])

            # Step 3: Clean each segment
            cleaned_segments = []
            for seg in expanded_segments:
                # Remove "in XYZ County" suffix
                cleaned = re.sub(r"\s+in\s+[\w\s]+County", "", seg, flags=re.IGNORECASE).strip()
                # Remove leading "and"
                cleaned = re.sub(r"^and\s+", "", cleaned, flags=re.IGNORECASE).strip()
                # Remove pure county references e.g. "Morris County"
                if re.match(r"^[A-Za-z\s]+County$", cleaned, re.IGNORECASE):
                    continue
                # Remove empty
                if not cleaned:
                    continue
                cleaned_segments.append(cleaned)

            # Step 4: Single clean non-RWD location → structured
            if len(cleaned_segments) == 1 and not NON_GEOCODABLE.search(cleaned_segments[0]):
                return cleaned_segments[0], "structured"

            # Step 5: Multiple locations → unstructured
            return cleaned_segments if len(cleaned_segments) > 1 else cleaned_segments[0], "unstructured"

    return None, "unmatched"

In [37]:

# def extract_city(title):
#     """
#     Look for 'City of' and return the very next word
#     """
#     match = re.search(r"City of\s+(\w+)", title)
#     return match.group(1) if match else None


def extract_county(title):
    """
    Look for 'County' and return the word immediately before it
    """
    match = re.search(r"(\w+)\s+County", title)
    return f"{match.group(1)} County" if match else None





In [41]:
import os
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# -------- Main Processing --------

INPUT_FOLDER = "extracted_json"
OUTPUT_FOLDER = "processed_json"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def process_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        title = item.get("title", "")
        posted_on = item.get("posted_on", "")
        issued_date = None
        rescinded_date = None
        if "Issued for" in title or "Re-issued for" in title:
            issued_date = extract_date(posted_on)
        if "Rescinded for" in title:
            rescinded_date = extract_date(posted_on)
        item["issued_date"] = issued_date
        item["rescinded_date"] = rescinded_date
        item["city"], item["city_type"] = extract_city(title)
        item["county"] = extract_county(title)
    output_path = Path(OUTPUT_FOLDER) / Path(file_path).name
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

# Process all JSON files in folder
for file_name in os.listdir(INPUT_FOLDER):
    if file_name.endswith(".json"):
        process_file(os.path.join(INPUT_FOLDER, file_name))

files = [
    os.path.join(INPUT_FOLDER, f)
    for f in os.listdir(INPUT_FOLDER)
    if f.endswith(".json")
]

with ThreadPoolExecutor(max_workers=4) as executor:
    executor.map(process_file, files)